In [ ]:
import pandas as pd
import os

# 분석에 사용할 원본 CSV 파일 경로를 지정
file_complaint = "../data/대구광역시 북구_불법주정차 민원_20211123.csv"

# 민원 파일은 CP949 인코딩으로 저장되어 있어서 cp949로 불러옵니다.
df = pd.read_csv(file_complaint, encoding="cp949")

# 데이터가 정상적으로 불러와졌는지 확인합니다.
print(df.head())

# 컬럼명과 데이터 크기를 확인합니다.
print("컬럼:", df.columns.tolist())
print("데이터 크기:", df.shape)

# 결측치가 있는지 확인합니다.
print(df.isnull().sum())

# 민원발생년도 범위와 연도별 행 수를 확인합니다.
print("연도 범위:", df["민원발생년도"].min(), "~", df["민원발생년도"].max())
print(df.groupby("민원발생년도").size())

# 경도/위도 값의 범위를 확인합니다.
# 대구의 위도는 약 35, 경도는 약 128 입니다.
print("경도 컬럼 범위:", df["경도"].min(), "~", df["경도"].max())
print("위도 컬럼 범위:", df["위도"].min(), "~", df["위도"].max())

# 경도 컬럼에 35대 값이 들어있으므로 두 컬럼 이름이 서로 바뀌어 있습니다.

# 바뀐 경도/위도 이름을 바로잡고, 긴 컬럼명은 짧게 바꿉니다.
df = df.rename(columns={
    "경도": "위도",
    "위도": "경도",
    "민원발생누적건수": "민원건수",
    "민원발생년도": "연도"
})

print(df.columns.tolist())

# 주소가 '대구광역시 북구'로 시작하지 않는 행을 확인합니다.
wrong_address = df[~df["주소"].str.startswith("대구광역시 북구 ")]
print("형식이 다른 주소 수:", len(wrong_address))
print(wrong_address["주소"].tolist())

# '대구광역시 산격동 ...' 처럼 '북구'가 빠진 주소에 '북구'를 넣어 형식을 통일합니다.
df["주소"] = df["주소"].str.replace(r"^대구광역시 (?!북구)", "대구광역시 북구 ", regex=True)

# 그래프에 쓰기 좋게 '대구광역시 북구 '를 뺀 짧은 주소 컬럼을 만듭니다.
df["짧은주소"] = df["주소"].str.replace("대구광역시 북구 ", "", regex=False)

print("형식이 다른 주소 수:", (~df["주소"].str.startswith("대구광역시 북구 ")).sum())

# 주소에서 동 이름을 뽑습니다.
# 산격3동 -> 산격동, 칠성동2가 / 노원동3가 -> 칠성동 / 노원동 으로 합칩니다.
dong = df["주소"].str.extract(r"([가-힣]+?)\d*(동)(?:\d가)?")
df["동"] = dong[0] + dong[1]

# 도로명 주소라서 동을 찾지 못한 행은 '미상'으로 표시합니다.
print("동을 찾지 못한 주소:", df.loc[df["동"].isnull(), "주소"].tolist())
df["동"] = df["동"].fillna("미상")

print(df["동"].value_counts())

# 같은 연도에 같은 주소가 여러 번 나오는지 확인합니다.
dup_count = df.duplicated(subset=["연도", "주소"]).sum()
print("같은 연도·주소가 반복되는 행 수:", dup_count)

# 같은 연도·주소는 민원 건수를 더해서 한 행으로 합칩니다.
df = df.groupby(["연도", "주소", "짧은주소", "동"], as_index=False).agg(
    민원건수=("민원건수", "sum"),
    위도=("위도", "first"),
    경도=("경도", "first")
)

print("합친 뒤 데이터 크기:", df.shape)

# 최종 전처리 결과를 확인합니다.
print("최종 데이터 크기:", df.shape)
print("연도 범위:", df["연도"].min(), "~", df["연도"].max())
print("전체 민원 건수:", df["민원건수"].sum())

# 앞부분 5행을 확인합니다.
display(df.head())



In [ ]:
# 전처리 결과를 저장할 폴더가 없으면 새로 만듭니다.
os.makedirs("../data/processed", exist_ok=True)

# utf-8-sig로 저장하면 엑셀에서 열었을 때도 한글이 깨질 가능성이 적습니다.
output_path = "../data/processed/lee_complaint_2018_2021.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("저장 완료:", output_path)

In [ ]:
# 그래프를 그리기 위한 matplotlib와 seaborn을 불러옵니다.
import matplotlib.pyplot as plt
import seaborn as sns

# 윈도우에서 그래프 한글이 깨지지 않도록 맑은 고딕을 사용합니다.
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 그래프를 저장할 폴더가 없으면 만듭니다.
os.makedirs("images", exist_ok=True)

In [ ]:
#데이터시각화시작

In [ ]:
# 연도별 민원 건수 합계와 민원이 들어온 주소(지점) 수를 계산합니다.
year_count = df.groupby("연도")["민원건수"].sum()
year_place = df.groupby("연도")["주소"].nunique()

year_summary = pd.DataFrame({
    "민원건수": year_count,
    "지점수": year_place
})
year_summary["지점당평균"] = (year_summary["민원건수"] / year_summary["지점수"]).round(1)
year_summary["전년대비증감률(%)"] = (year_summary["민원건수"].pct_change() * 100).round(1)

# 계산 결과를 확인합니다.
display(year_summary)

In [ ]:
# 연도별 변화는 시간의 흐름이 중요하므로 선그래프로 나타냅니다.
plt.figure(figsize=(9, 5))
plt.plot(year_count.index, year_count.values, marker="o")

# 점 위에 건수를 표시합니다.
for year, count in year_count.items():
    plt.text(year, count, f"{count:,}", ha="center", va="bottom")

plt.title("2018~2021년 연도별 불법주정차 민원 발생 건수 (2021년은 11/23까지)")
plt.xlabel("연도")
plt.ylabel("민원 건수")
plt.xticks(year_count.index)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("images/lee_complaint_year_count.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 연도마다 민원 건수가 많은 순으로 순위를 매깁니다.
# 건수가 같으면 같은 순위(min)를 줍니다.
df["연도내순위"] = df.groupby("연도")["민원건수"].rank(method="min", ascending=False).astype(int)

# 연도별 TOP 10 주소를 표로 만듭니다.
top_n = 10
year_top_address = (
    df[df["연도내순위"] <= top_n]
    .sort_values(["연도", "연도내순위"])
    [["연도", "연도내순위", "짧은주소", "동", "민원건수"]]
)

#display(year_top_address)


# 연도별 TOP 10 주소를 한눈에 비교하도록 연도 x 순위 표로 바꿉니다.
year_top_table = year_top_address.drop_duplicates(["연도", "연도내순위"]).pivot(
    index="연도내순위", columns="연도", values="짧은주소"
)

display(year_top_table)

In [ ]:
# 연도별 TOP 10 주소를 4개의 막대그래프로 나란히 그립니다.
years = sorted(df["연도"].unique())
fig, axes = plt.subplots(1, len(years), figsize=(20, 6))

for ax, year in zip(axes, years):
    top = df[df["연도"] == year].nlargest(top_n, "민원건수")
    ax.barh(top["짧은주소"], top["민원건수"])
    ax.invert_yaxis()
    ax.set_title(f"{year}년 민원 TOP {top_n}")
    ax.set_xlabel("민원 건수")

plt.suptitle("연도별 불법주정차 민원 많은 주소 TOP 10", fontsize=15)
plt.tight_layout()
plt.savefig("images/lee_complaint_year_top_address.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 동 x 연도별 민원 건수를 교차 집계합니다. (미상 제외)
dong_year = df[df["동"] != "미상"].groupby(["동", "연도"])["민원건수"].sum().unstack(fill_value=0)
dong_year["합계"] = dong_year.sum(axis=1)
dong_year = dong_year.sort_values("합계", ascending=False)

display(dong_year)

In [ ]:
# 연도마다 동 순위를 계산합니다. (1위 = 민원이 가장 많은 동)
dong_rank = dong_year.drop(columns="합계").rank(ascending=False, method="min").astype(int)

# '합계' 컬럼을 빼고 나면 연도 컬럼을 숫자형으로 바꿔 줍니다. (그래프 x축에 쓰기 위해)
dong_rank.columns = dong_rank.columns.astype(int)

display(dong_rank.head(10))

In [ ]:
# 전체 기간 동별 민원 건수를 막대그래프로 비교합니다.
dong_total = dong_year["합계"]

plt.figure(figsize=(12, 5))
plt.bar(dong_total.index, dong_total.values)
plt.title("2018~2021년 동별 불법주정차 민원 건수")
plt.xlabel("동")
plt.ylabel("민원 건수")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("images/lee_complaint_dong_total.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 동 x 연도 패턴을 한눈에 보기 위해 히트맵으로 나타냅니다.
plt.figure(figsize=(8, 10))
sns.heatmap(dong_year.drop(columns="합계"), cmap="YlOrRd", annot=True, fmt="d")
plt.title("동 × 연도 불법주정차 민원 건수")
plt.xlabel("연도")
plt.ylabel("동")
plt.tight_layout()
plt.savefig("images/lee_complaint_dong_year_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 전체 상위 8개 동의 연도별 순위 변화를 선그래프로 나타냅니다.
top_dongs = dong_year.head(8).index

plt.figure(figsize=(9, 6))
for dong_name in top_dongs:
    plt.plot(dong_rank.columns, dong_rank.loc[dong_name], marker="o", label=dong_name)

plt.gca().invert_yaxis()   # 1위가 위로 오도록 뒤집습니다.
plt.title("상위 8개 동의 연도별 민원 순위 변화")
plt.xlabel("연도")
plt.ylabel("순위")
plt.xticks(dong_rank.columns)
plt.yticks(range(1, dong_rank.loc[top_dongs].values.max() + 1))
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("images/lee_complaint_dong_rank_change.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 주소별 2018~2021 민원 합계와 민원이 들어온 연도 수를 계산합니다.
address_total = df.groupby(["주소", "짧은주소", "동"], as_index=False).agg(
    민원합계=("민원건수", "sum"),
    발생연도수=("연도", "nunique"),
    위도=("위도", "first"),
    경도=("경도", "first")
).sort_values("민원합계", ascending=False).reset_index(drop=True)

address_total.index = address_total.index + 1   # 1위부터 시작
address_total.index.name = "전체순위"

# 4년 내내 민원이 들어온 주소 수를 확인합니다.
print("4년 연속 민원 발생 주소 수:", (address_total["발생연도수"] == 4).sum())

#display(address_total.head(20))

# 전체 기간 민원 TOP 20 주소를 가로 막대그래프로 나타냅니다.
top20 = address_total.head(20)

plt.figure(figsize=(10, 7))
plt.barh(top20["짧은주소"], top20["민원합계"])
plt.gca().invert_yaxis()
plt.title("2018~2021년 불법주정차 민원 TOP 20 주소")
plt.xlabel("민원 건수")
plt.ylabel("주소")
plt.tight_layout()
plt.savefig("images/lee_complaint_top20_address.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 민원 발생 위치를 좌표로 찍어 봅니다. (점 크기와 색 = 민원 건수)
map_data = address_total.sort_values("민원합계")   # 큰 점이 위에 그려지도록 정렬

plt.figure(figsize=(9, 9))
plt.scatter(
    map_data["경도"], map_data["위도"],
    s=map_data["민원합계"] / 5 + 2,
    c=map_data["민원합계"], cmap="YlOrRd", alpha=0.7
)
plt.colorbar(label="민원 건수")

# TOP 5 주소는 이름을 표시합니다.
for _, row in address_total.head(5).iterrows():
    plt.annotate(row["짧은주소"], (row["경도"], row["위도"]), fontsize=9,
                 xytext=(5, 5), textcoords="offset points")

plt.title("2018~2021년 불법주정차 민원 발생 위치 (대구 북구)")
plt.xlabel("경도")
plt.ylabel("위도")
plt.tight_layout()
plt.savefig("images/lee_complaint_map.png", dpi=150, bbox_inches="tight")
plt.show()